In [1]:
import cv2
import numpy as np

print(f"OpenCV Version: {cv2.__version__}")

try:
    count = cv2.cuda.getCudaEnabledDeviceCount()
    if count > 0:
        print(f"Success! {count} CUDA device(s) detected.")
    else:
        print("No CUDA devices detected. (This is expected for Iris Xe)")
except AttributeError:
    print("CUDA module not found in this OpenCV build. System will default to CPU mode.")

OpenCV Version: 4.11.0
No CUDA devices detected. (This is expected for Iris Xe)


In [2]:
import cv2
import numpy as np
from threading import Thread
import time

class VideoGet:
    """
    Class that continuously gets frames from a VideoCapture object
    with a dedicated thread.
    """
    def __init__(self, src=0):
        # src=0 is usually the default webcam
        self.stream = cv2.VideoCapture(src)
        (self.grabbed, self.frame) = self.stream.read()
        self.stopped = False

    def start(self):    
        # Start the thread to read frames from the video stream
        Thread(target=self.update, args=()).start()
        return self

    def update(self):
        # Keep looping until the thread is stopped
        while not self.stopped:
            if not self.grabbed:
                self.stop()
            else:
                # Read the next frame from the stream
                (self.grabbed, self.frame) = self.stream.read()

    def read(self):
        # Return the most recent frame
        return self.frame

    def stop(self):
        self.stopped = True
        self.stream.release()

In [3]:
# Initialize the threaded video getter
video_getter = VideoGet(0).start()

# Give the camera a second to warm up
time.sleep(1.0)

try:
    while True:
        # Get the latest frame
        frame = video_getter.read()
        
        if frame is not None:
            cv2.imshow("VideoGet Test", frame)
        
        # Check for 'q' key press to exit
        # Wait for 1 ms
        if cv2.waitKey(1) == ord("q"):
            break
finally:
    # Always ensure we stop the thread and close windows
    # even if an error occurs
    video_getter.stop()
    cv2.destroyAllWindows()
    print("Camera stopped and windows closed.")

Camera stopped and windows closed.


In [4]:
def process_frame_cpu(frame):
    """
    CPU-Optimized Cartoonifier Pipeline
    Target: 30 FPS on Integrated Graphics
    """
    # --- 1. Downscale (Critical for CPU speed) ---
    height, width = frame.shape[:2]
    target_height = 480
    # Calculate width to keep aspect ratio
    aspect_ratio = width / height
    target_width = int(target_height * aspect_ratio)
    
    # Resize input
    small_frame = cv2.resize(frame, (target_width, target_height), interpolation=cv2.INTER_LINEAR)

    # --- 2. Color Smoothing (The "Painted" Look) ---
    # CPU Strategy: Small 'd' value, repeated twice is often faster/better looking 
    # than one giant filter pass.
    color = small_frame
    for _ in range(2):
        color = cv2.bilateralFilter(color, d=5, sigmaColor=50, sigmaSpace=50)

    # --- 3. Edge Detection (The "Ink" Lines) ---
    # Convert to grayscale
    gray = cv2.cvtColor(small_frame, cv2.COLOR_BGR2GRAY)
    # Median blur to remove noise (speckles) before finding edges
    gray_blur = cv2.medianBlur(gray, 5)
    
    # Adaptive Threshold creates the thick black outlines
    # blockSize=9, C=5 are tuned for good contrast
    edges = cv2.adaptiveThreshold(
        gray_blur, 
        255, 
        cv2.ADAPTIVE_THRESH_MEAN_C, 
        cv2.THRESH_BINARY, 
        blockSize=9, 
        C=5
    )

    # --- 4. Synthesis ---
    # The 'edges' image is black and white (1 channel).
    # We need to convert it to color (3 channels) to merge it.
    edges_color = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
    
    # Combine: keep color where edges are white; turn black where edges are black
    cartoon = cv2.bitwise_and(color, edges_color)

    return cartoon

In [5]:
# 1. Start the camera thread
video_getter = VideoGet(0).start()
# Allow camera to warm up
time.sleep(1.0)

# Variables for FPS calculation
prev_frame_time = 0
new_frame_time = 0

try:
    while True:
        # 1. Get Frame (Instant, from thread)
        frame = video_getter.read()
        
        if frame is None:
            continue

        # 2. Process Frame (The CPU Cartoon Logic)
        cartoon_frame = process_frame_cpu(frame)

        # 3. Calculate FPS
        new_frame_time = time.time()
        # Avoid division by zero
        fps = 1 / (new_frame_time - prev_frame_time + 1e-5) 
        prev_frame_time = new_frame_time
        
        # 4. Draw FPS on screen
        cv2.putText(
            cartoon_frame, 
            f"FPS: {int(fps)} (CPU Mode)", 
            (10, 30), 
            cv2.FONT_HERSHEY_SIMPLEX, 
            0.7, 
            (0, 255, 0), 
            2
        )

        # 5. Show Result
        cv2.imshow("Real-Time Cartoonifier", cartoon_frame)

        # 6. Check for Exit
        if cv2.waitKey(1) == ord("q"):
            break

finally:
    # Clean exit
    video_getter.stop()
    cv2.destroyAllWindows()
    print("Cartoonifier stopped.")

Cartoonifier stopped.
